In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("data/clean/rtt_clean.csv", parse_dates=["month"])
con = sqlite3.connect(":memory:")
df.to_sql("rtt", con, index=False)

def sql(query):
    return pd.read_sql(query, con)

In [2]:
sql("""
SELECT month,
       SUM(waiting_list) AS waiting_list,
       ROUND(100.0 * SUM(within_18wks) / SUM(known_start), 1) AS pct_within_18wks,
       SUM(over_52wks) AS over_52wks
FROM rtt
GROUP BY month
""")

,month,waiting_list,pct_within_18wks,over_52wks
0,2025-04-01 00:00:00,7421453,59.7,192560.0
1,2026-03-01 00:00:00,7045510,65.2,95824.0


In [3]:
sql("""
SELECT specialty,
       ROUND(100.0 * SUM(CASE WHEN month LIKE '2025-04%' THEN within_18wks END)
                    / SUM(CASE WHEN month LIKE '2025-04%' THEN known_start END), 1) AS apr_2025,
       ROUND(100.0 * SUM(CASE WHEN month LIKE '2026-03%' THEN within_18wks END)
                    / SUM(CASE WHEN month LIKE '2026-03%' THEN known_start END), 1) AS mar_2026
FROM rtt
GROUP BY specialty
ORDER BY mar_2026
""")

,specialty,apr_2025,mar_2026
0,Oral Surgery Service,50.4,55.6
1,Plastic Surgery Service,53.9,56.6
2,Ear Nose and Throat Service,50.9,58.4
3,Trauma and Orthopaedic Service,56.4,59.1
4,Neurology Service,53.8,60.6
5,Gynaecology Service,55.1,60.8
6,Neurosurgical Service,58.8,62.2
7,General Surgery Service,59.1,62.9
8,Dermatology Service,60.6,65.0
9,Other - Surgical Services,61.2,66.1


In [4]:
sql("""
WITH apr AS (
    SELECT org_code, MAX(trust) AS trust, SUM(over_52wks) AS apr_52wks
    FROM rtt WHERE month LIKE '2025-04%' AND provider_type = 'NHS trust'
    GROUP BY org_code
),
mar AS (
    SELECT org_code, SUM(over_52wks) AS mar_52wks, SUM(waiting_list) AS mar_list
    FROM rtt WHERE month LIKE '2026-03%' AND provider_type = 'NHS trust'
    GROUP BY org_code
)
SELECT a.trust, a.apr_52wks, m.mar_52wks, m.mar_52wks - a.apr_52wks AS change
FROM apr a
INNER JOIN mar m ON a.org_code = m.org_code
WHERE m.mar_list >= 10000
ORDER BY change
LIMIT 10
""")

,trust,apr_52wks,mar_52wks,change
0,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,8443.0,1534.0,-6909.0
1,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,5199.0,397.0,-4802.0
2,UNIVERSITY HOSPITALS SUSSEX NHS FOUNDATION TRUST,6925.0,3580.0,-3345.0
3,EAST SUFFOLK AND NORTH ESSEX NHS FOUNDATION TRUST,3233.0,276.0,-2957.0
4,BARTS HEALTH NHS TRUST,3678.0,1008.0,-2670.0
5,NORTH WEST ANGLIA NHS FOUNDATION TRUST,2977.0,408.0,-2569.0
6,ROYAL FREE LONDON NHS FOUNDATION TRUST,3899.0,1331.0,-2568.0
7,FRIMLEY HEALTH NHS FOUNDATION TRUST,3110.0,604.0,-2506.0
8,LIVERPOOL UNIVERSITY HOSPITALS NHS FOUNDATION ...,3019.0,560.0,-2459.0
9,BEDFORDSHIRE HOSPITALS NHS FOUNDATION TRUST,2781.0,453.0,-2328.0


## SQL cross-check
National figures recalculated in SQL match pandas. Additional queries: specialty performance in April 2025 vs March 2026 using conditional aggregation (CASE WHEN inside SUM), and the trusts with the largest reduction in 52-week waits using two CTEs joined on organisation code (INNER JOIN).